# Vision basics

**Authors:** Sibo Wang-Chen, Thomas Ka Chung Lam

**Summary:** In this tutorial, we will build a simple model to control the fly to follow a moving sphere. By doing so, we will also demonstrate how one can create a custom arena.

Animals typically navigate over rugged terrain to reach attractive objects (e.g., potential mates, food sources) and to avoid repulsive features (e.g., pheromones from predators) and obstacles. Animals use a hierarchical controller to achieve these goals: processing higher-order sensory signals, using them to select the next action, and translating these decisions into descending commands that drive lower-level motor systems. We aimed to simulate this sensorimotor hierarchy by adding vision and olfaction to NeuroMechFly.

## Retina simulation

A fly's compound eye consists of ~700–750 individual units called ommatidia arranged in a hexagonal pattern (see the left panel of the figure below from the [droso4schools project](https://droso4schools.wordpress.com/l4-enzymes/#5); see also [this article](https://azretina.sites.arizona.edu/node/789) from the Arizona Retina Project). To emulate this, we attached a color camera to each of our model's compound eyes (top right panel). We then transformed each camera image into 721 bins, representing ommatidia. Based on previous studies, we assume a 270° combined azimuth for the fly's field of view, with a ~17° binocular overlap. Visual sensitivity has evolved to highlight ethologically relevant color spectra at different locations in the environment. Here, as an initial step toward enabling this heterogeneity in our model, we implemented yellow- and pale-type ommatidia — sensitive to the green and blue channels of images rendered by the physics simulator — randomly assigned at a 7:3 ratio (as reported in [Rister et al, 2013](https://pubmed.ncbi.nlm.nih.gov/23293281/)). Users can substitute the green and blue channel values with the desired light intensities sensed by yellow- and pale-type ommatidia to achieve more biorealistic chromatic vision.

<img src="https://raw.githubusercontent.com/NeLy-EPFL/_media/refs/heads/main/flygym/vision_basics/vision.png" width="800"/>

In NeuroMechFly, the main interface to interact with the fly's vision is the [Retina class](https://neuromechfly.org/api_ref/vision.html#retina-simulation). It is embedded into the fly as its `.retina` attribute when `enable_vision` is set to True. Let's reproduce the top right panel of the figure above by placing the fly in an arena with three pillars and simulating its visual experience.

To start, we do the necessary imports:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange
import dm_control.mjcf as mjcf

from flygym import Simulation
from flygym.compose import ActuatorType, Fly
from flygym.compose.world import FlatGroundWorld, ObstaclesMixin
from flygym.examples.locomotion import TurningController
from flygym.utils.math import Rotation3D
from utils import create_fly, create_simulation, show_video


class ObstacleWorld(ObstaclesMixin, FlatGroundWorld):
    pass


obstacle_positions = [(7.5, 0, 2), (12.5, 5, 2), (17.5, -5, 2)]
obstacle_rgba = [(0.14, 0.14, 0.2, 1), (0.2, 0.8, 0.2, 1), (0.2, 0.2, 0.8, 1)]

world = ObstacleWorld()
for pos, rgba in zip(obstacle_positions, obstacle_rgba):
    world.add_obstacle(pos=pos, rgba=rgba, type="cylinder")

camera_kwargs = {
    "mode": "fixed",
    "pos": (13, -18, 9),
    "rotation": Rotation3D("euler", (np.deg2rad(65), 0, 0)),
    "fovy": 45,
}

fly = create_fly(enable_vision=True)
sim = create_simulation(
    world=world,
    fly=fly,
    spawn_position=(13, -5, 0.2),
    spawn_rotation=Rotation3D(
        "quat", (0.8191520442889918, 0.0, 0.0, 0.573576436351046)
    ),
    camera_kwargs=camera_kwargs,
)
sim.step()
sim.render_as_needed()
show_video(sim)

ModuleNotFoundError: No module named 'flygym'

We can access the intensities sensed by the fly's ommatidia from the simulation:

In [ ]:
ommatidia_readouts = sim.get_ommatidia_readouts(fly.name)
print(f"{ommatidia_readouts.shape=}")
print(f"{ommatidia_readouts.dtype=}")

This gives us a ``(2, 721, 2)`` array representing the light intensities sensed by the ommatidia. The values are normalized to [0, 1]. The first dimension is for the two eyes (left and right). The second dimension is for the 721 ommatidia per eye. The third dimension is for the two color channels (yellow- and pale-type). For each ommatidium, only one of the two channels is nonzero. The yellow- and pale-type ommatidia are split at a 7:3 ratio. We can verify this below:

In [ ]:
nonzero_idx = np.nonzero(ommatidia_readouts)
unique_vals, val_counts = np.unique(nonzero_idx[2], return_counts=True)
val_counts / val_counts.sum()

This array representation is not ideal for visualization. We can use the `hex_pxls_to_human_readable` method of the retina to convert it into a standard 8-bit RGB image. We set `color_8bit=True` for efficiency and to return values in the [0, 255] range. We further take the grayscale image (disregarding yellow- vs. pale-type) by taking the maximum along the last dimension (color channels).

In [ ]:
vision_left = fly.retina.hex_pxls_to_human_readable(
    ommatidia_readouts[0], color_8bit=True
).max(axis=-1)
vision_right = fly.retina.hex_pxls_to_human_readable(
    ommatidia_readouts[1], color_8bit=True
).max(axis=-1)

fig, axs = plt.subplots(1, 2, figsize=(6, 3), tight_layout=True)
axs[0].imshow(vision_left, cmap="gray", vmin=0, vmax=255)
axs[0].axis("off")
axs[0].set_title("Left eye")
axs[1].imshow(vision_right, cmap="gray", vmin=0, vmax=255)
axs[1].axis("off")
axs[1].set_title("Right eye")

We can also access the raw RGB vision (before pixels are binned into ommatidia) through `sim.get_raw_vision`:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
for i_eye in range(2):
    axs[i_eye].imshow(sim.get_raw_vision(fly.name)[i_eye])

## A dynamic arena with a moving sphere
The next step is to create a custom arena with a moving sphere. To do this, we will implement a `MovingObjectMixin` that can be mixed into any world to add moving objects. The `MovingObjectMixin` will have a method `add_object` that allows us to add a moving object to the world, and a method `step` that updates the position of the moving objects at each timestep.

In [ ]:
class MovingObjectMixin:
    """Mixin that adds moving objects (e.g., spheres) to any world."""

    mjcf_root: mjcf.RootElement

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.object_names = []
        self.lateral_magnitudes = np.empty(0)
        self.periods = np.empty(0)
        self.speeds = np.empty(0)

    def add_object(
        self,
        lateral_magnitude=2,
        speed=10,
        period=2,
        pos=(5, 0, 1),
        type="sphere",
        size=(1, 1),
        rgba=(0, 0, 0, 1),
        name=None,
        **kwargs,
    ):
        """Add a moving object to the arena.

        Parameters
        ----------
        lateral_magnitude : float
            Amplitude of the lateral sinusoidal motion.
        speed : float
            Forward speed of the object (mm/s).
        period : float
            Period of the lateral oscillation (seconds).
        pos : tuple
            Initial (x, y, z) position.
        type : str
            MuJoCo geom type (e.g., ``"sphere"``).
        size : tuple
            Size parameters for the geom.
        rgba : tuple
            Color of the object (RGBA, each in [0, 1]).
        name : str, optional
            Unique name for the object.
        """
        if name is None:
            name = f"object_{len(self.object_names)}"
        assert name not in self.object_names, f"Object with name {name} already exists"
        ball_body = self.mjcf_root.worldbody.add(
            "body",
            name=f"{name}_mocap",
            mocap=True,
            pos=pos,
            gravcomp=1,
        )
        ball_body.add(
            "geom",
            name=f"{name}_geom",
            type=type,
            size=size,
            rgba=rgba,
            **kwargs,
        )
        self.object_names.append(name)
        self.lateral_magnitudes = np.append(self.lateral_magnitudes, lateral_magnitude)
        self.periods = np.append(self.periods, period)
        self.speeds = np.append(self.speeds, speed)

    def step(self, sim: Simulation):
        """Update the positions of all moving objects."""
        mocap_ids = np.array(
            [
                sim.mj_model.body(f"{name}_mocap").mocapid.item()
                for name in self.object_names
            ]
        )
        heading_vecs = np.column_stack(
            [
                np.ones(len(self.object_names)),
                self.lateral_magnitudes
                * np.cos(sim.mj_data.time / self.periods * 2 * np.pi),
            ]
        )
        heading_vecs /= np.linalg.norm(heading_vecs, axis=-1, keepdims=True)
        sim.mj_data.mocap_pos[mocap_ids, :2] += (
            self.speeds[:, None] * heading_vecs * sim.timestep
        )


class MovingObjectWorld(MovingObjectMixin, FlatGroundWorld):
    pass


world = MovingObjectWorld()
world.add_object()
fly = create_fly(enable_vision=True)
sim = create_simulation(world=world, fly=fly, camera_kwargs={"pos": (20, 0, 40)})

for i in trange(20000):
    sim.step()
    world.step(sim)
    sim.render_as_needed()

show_video(sim)

## Visual feature preprocessing

We preprocess the visual input by computing the x–y position of the object on the retina along with its size relative to the whole retinal image. We do this by applying binary thresholding to the image and calculating its size and center of mass. Any pixel darker than the threshold will be considered part of the black sphere. We also define a decision interval $\tau$: the turning signal is recomputed every $\tau$ seconds. We pre-compute the center of mass (COM) of all ommatidia so that when we need the COM of the object (a masked subset of pixels), we can simply average the COMs of the selected pixels.

## Implementing an object-tracking controller

Now that we have implemented the arena, we define the controller logic that outputs the 2D descending representation based on the extracted visual features. As a proof of concept, we have hand-tuned the following relationship:

$$
\delta_i = \begin{cases}
\min(\max(k\, a_i + b,\; \delta_\text{min}),\; \delta_\text{max})   & \text{if } A_i > A_\text{thr} \\
1  & \text{otherwise}
\end{cases}
$$

where $\delta_i$ is the descending modulation signal on side $i$; $a_i$ is the azimuth of the object expressed as the deviation from the anterior edge of the eye's field of view, normalized by the horizontal field of view of the retina; $A_i$ is the relative size of the object on the retina; $k = -3$, $b = 1$ are parameters describing the response curve; $\delta_\text{min} = 0.2$, $\delta_\text{max} = 1$ describe the range of the descending signal; and $A_\text{thr} = 1\%$ is the threshold below which the object is considered unseen.

In [ ]:
class VisualTaxisController(TurningController):
    """Controller that steers the fly toward a dark object using visual taxis."""

    def __init__(
        self,
        sim: Simulation,
        fly: Fly,
        obj_threshold=0.15,
        decision_interval=0.15,
        **kwargs,
    ):
        super().__init__(timestep=sim.timestep, **kwargs)
        self.obj_threshold = obj_threshold
        self.sim = sim
        self.fly = fly
        self._last_decision_time = -np.inf
        self.decision_interval = decision_interval
        self.descending_signals = np.array([1.0, 1.0])

        # Pre-compute ommatidium center-of-mass positions
        self.coms = np.empty((fly.retina.num_ommatidia_per_eye, 2))
        for i in range(fly.retina.num_ommatidia_per_eye):
            mask = fly.retina.ommatidia_id_map == i + 1
            self.coms[i] = np.argwhere(mask).mean(axis=0)

    def _process_visual_observation(self, ommatidia_readouts):
        """Extract object position and size features from each eye."""
        features = np.zeros((2, 3))
        for i, ommatidia_readings in enumerate(ommatidia_readouts):
            is_obj = ommatidia_readings.max(axis=1) < self.obj_threshold
            obj_coords = self.coms[is_obj]
            if obj_coords.shape[0] > 0:
                features[i, :2] = obj_coords.mean(axis=0)
            features[i, 2] = obj_coords.shape[0]
        features[:, 0] /= self.fly.retina.nrows  # normalize y_center
        features[:, 1] /= self.fly.retina.ncols  # normalize x_center
        features[:, 2] /= self.fly.retina.num_ommatidia_per_eye  # normalize area
        return features.ravel()

    @staticmethod
    def _calc_ipsilateral_speed(deviation, is_found):
        if not is_found:
            return 1.0
        return np.clip(1 - deviation * 3, 0.4, 1.2)

    def _update_descending_signals(self, preprocessed_features):
        left_found = preprocessed_features[2] > 0.01
        left_deviation = 1 - preprocessed_features[1]
        self.descending_signals[0] = self._calc_ipsilateral_speed(
            left_deviation, left_found
        )

        right_found = preprocessed_features[5] > 0.01
        right_deviation = preprocessed_features[4]
        self.descending_signals[1] = self._calc_ipsilateral_speed(
            right_deviation, right_found
        )

    def step(self):
        if self.sim.time - self._last_decision_time >= self.decision_interval:
            ommatidia_readouts = self.sim.get_ommatidia_readouts(self.fly.name)
            features = self._process_visual_observation(ommatidia_readouts)
            self._update_descending_signals(features)
            self._last_decision_time = self.sim.time
        return super().step(self.descending_signals)

In [ ]:
world = MovingObjectWorld()
world.add_object()
fly = create_fly(enable_vision=True)
sim = create_simulation(world=world, fly=fly, camera_kwargs={"pos": (20, 0, 40)})
controller = VisualTaxisController(sim=sim, fly=fly)

for i in trange(40000):
    world.step(sim)
    sim.step()
    joint_angles, adhesion = controller.step()
    sim.set_actuator_inputs(fly.name, ActuatorType.POSITION, joint_angles)
    sim.set_actuator_inputs(fly.name, ActuatorType.ADHESION, adhesion)
    sim.render_as_needed()

show_video(sim)